# Programming Assignment: Building Clustering Models from Scratch

Welcome to the assignment for Clustering Algorithms.

Clustering methods are among the most popular and powerful unsupervised machine learning techniques for discovering hidden patterns in data. From simple algorithms like K-Means to more advanced methods such as Gaussian Mixture Models and Hierarchical Clustering, these approaches are widely used in real-world applications including customer segmentation, anomaly detection, document grouping, and exploratory data analysis.

In this assignment, you'll implement clustering algorithms using both scikit-learn and from-scratch implementations using NumPy. You'll explore hard and soft clustering techniques, understand their assumptions and limitations, and then learn how different clustering strategies group data based on distance, similarity, or probability distributions.

Clustering algorithms work by grouping data points such that points within the same cluster are more similar to each other than to those in other clusters. Unlike supervised learning, no labels are provided, so evaluating results requires specialized metrics and careful analysis. While some clustering methods are sensitive to initialization and parameter choices, combining evaluation techniques and probabilistic modeling helps create more robust and meaningful groupings.

**What You Will Do in This Assignment**

* Understand the mathematical foundations of clustering including distance metrics, centroids, and similarity measures.
* Build and visualize K-Means clustering models, interpreting cluster assignments.
* Select the optimal number of clusters using the elbow method and silhouette analysis.
* Apply Gaussian Mixture Models for soft clustering with probability-based assignments.
* Build Hierarchical Clustering models and visualize relationships using dendrograms.
* Evaluate clustering quality using internal and external metrics.
* Build a complete K-Means algorithm from scratch with initialization, assignment, and update steps.
* Implement Gaussian Mixture Models from scratch using the Expectation-Maximization algorithm.

Let's get started!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents

1. [Introduction to Clustering](#1)
2. [K-Means Clustering](#2)
   - [Exercise 1: Implement K-Means Clustering](#ex-1)
3. [Optimal K Selection](#3)
   - [Exercise 2: Elbow Method and Silhouette Analysis](#ex-2)
4. [Gaussian Mixture Models](#4)
   - [Exercise 3: Implement GMM Clustering](#ex-3)
5. [Hierarchical Clustering](#5)
   - [Exercise 4: Build Dendrogram and Compare Strategies](#ex-4)
6. [Clustering Evaluation](#6)
   - [Exercise 5: Apply Clustering Metrics](#ex-5)
7. [From-Scratch Implementations](#7)
   - [Exercise 6: K-Means from Scratch](#ex-6)
   - [Exercise 7: GMM from Scratch](#ex-7)
8. [Conclusion](#8)

<a name='1'></a>
## 1 - Introduction to Clustering

**Clustering** is an unsupervised learning technique that groups similar data points together without using labeled data. It's widely used for:

- **Customer Segmentation**: Grouping customers by purchasing behavior
- **Image Compression**: Reducing colors by clustering similar pixels
- **Anomaly Detection**: Identifying outliers as points far from clusters
- **Document Organization**: Grouping similar documents or articles
- **Biology**: Discovering species or gene expression patterns

### Types of Clustering:

1. **Partitional Clustering** (K-Means): Divides data into non-overlapping clusters
2. **Probabilistic Clustering** (GMM): Assigns soft probabilities to cluster membership
3. **Hierarchical Clustering**: Builds tree-like cluster structures
4. **Density-Based Clustering** (DBSCAN): Groups points in high-density regions

### Key Concepts:

**Inertia**: Sum of squared distances from points to their cluster centers
$$\text{Inertia} = \sum_{i=1}^{n} \min_{\mu_j \in C} ||x_i - \mu_j||^2$$

**Silhouette Score**: Measures how well-separated clusters are (-1 to 1, higher is better)
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

where $a(i)$ is average distance within cluster, $b(i)$ is average distance to nearest cluster.

### Import Required Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests

from sklearn.datasets import make_blobs, make_moons, load_iris, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score, normalized_mutual_info_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist
from scipy.stats import multivariate_normal
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("All libraries imported successfully!")

### Helper Functions

We'll define some helper functions for visualization and evaluation.

In [ ]:
def plot_clusters(X, labels, centers=None, title="Cluster Assignments"):
    """Plot 2D clustering results."""
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', 
                         alpha=0.6, edgecolors='k', s=50)
    
    if centers is not None:
        plt.scatter(centers[:, 0], centers[:, 1], 
                   c='red', marker='X', s=300, 
                   edgecolors='black', linewidths=2, 
                   label='Centroids')
        plt.legend()
    
    plt.colorbar(scatter, label='Cluster')
    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.tight_layout()
    plt.show()


def plot_silhouette(X, labels, n_clusters):
    """Plot silhouette analysis."""
    silhouette_vals = silhouette_samples(X, labels)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    y_lower = 10
    
    for i in range(n_clusters):
        cluster_silhouette_vals = silhouette_vals[labels == i]
        cluster_silhouette_vals.sort()
        
        size_cluster_i = cluster_silhouette_vals.shape[0]
        y_upper = y_lower + size_cluster_i
        
        color = plt.cm.viridis(float(i) / n_clusters)
        ax.fill_betweenx(np.arange(y_lower, y_upper),
                         0, cluster_silhouette_vals,
                         facecolor=color, edgecolor=color, alpha=0.7)
        
        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10
    
    ax.set_xlabel("Silhouette Coefficient")
    ax.set_ylabel("Cluster")
    ax.axvline(x=silhouette_score(X, labels), color="red", linestyle="--", 
               label=f"Average: {silhouette_score(X, labels):.3f}")
    ax.set_title(f"Silhouette Plot for {n_clusters} Clusters")
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_gmm_ellipses(gmm, X, labels):
    """Plot GMM clusters with confidence ellipses."""
    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, gmm.n_components))
    
    for i in range(gmm.n_components):
        # Plot points
        mask = labels == i
        plt.scatter(X[mask, 0], X[mask, 1], c=[colors[i]], 
                   alpha=0.4, s=30, edgecolors='k', linewidths=0.5)
        
        # Plot ellipse
        mean = gmm.means_[i]
        covar = gmm.covariances_[i]
        
        v, w = np.linalg.eigh(covar)
        v = 2.0 * np.sqrt(2.0) * np.sqrt(v)
        u = w[0] / np.linalg.norm(w[0])
        
        angle = np.arctan(u[1] / u[0])
        angle = 180.0 * angle / np.pi
        
        ell = plt.matplotlib.patches.Ellipse(
            mean, v[0], v[1], angle=180.0 + angle, 
            color=colors[i], alpha=0.3, linewidth=2
        )
        plt.gca().add_patch(ell)
        plt.scatter(mean[0], mean[1], c=[colors[i]], marker='X', 
                   s=200, edgecolors='black', linewidths=2)
    
    plt.title("GMM Clustering with Confidence Ellipses")
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.tight_layout()
    plt.show()

print("Helper functions defined!")

### Load and Prepare Data

In [ ]:
# Create synthetic dataset with clear clusters
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, 
                              n_features=2, cluster_std=0.60, 
                              random_state=42)

# Create non-convex dataset (moons)
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)

# Load Iris dataset for evaluation
iris = load_iris()
X_iris = iris.data[:, [2, 3]]  # Use petal dimensions
y_iris = iris.target

# Standardize features
scaler = StandardScaler()
X_blobs_scaled = scaler.fit_transform(X_blobs)
X_moons_scaled = scaler.fit_transform(X_moons)
X_iris_scaled = scaler.fit_transform(X_iris)

print(f"Blobs dataset shape: {X_blobs.shape}")
print(f"Moons dataset shape: {X_moons.shape}")
print(f"Iris dataset shape: {X_iris.shape}")
print(f"\nIris has {len(np.unique(y_iris))} true classes")

<a name='2'></a>
## 2 - K-Means Clustering

**K-Means** is one of the most popular clustering algorithms. It partitions data into K clusters by:

1. **Initialize**: Randomly select K cluster centers (centroids)
2. **Assignment**: Assign each point to the nearest centroid
3. **Update**: Recompute centroids as the mean of assigned points
4. **Repeat** steps 2-3 until convergence

### Algorithm:

**Assignment Step**:
$$c_i = \arg\min_{j} ||x_i - \mu_j||^2$$

**Update Step**:
$$\mu_j = \frac{1}{|C_j|} \sum_{x_i \in C_j} x_i$$

**Objective** (minimize inertia):
$$J = \sum_{i=1}^{n} ||x_i - \mu_{c_i}||^2$$

### Properties:

**Advantages**:
- Simple and fast
- Scales well to large datasets
- Guaranteed to converge

❌ **Limitations**:
- Must specify K in advance
- Sensitive to initialization
- Assumes spherical clusters
- Sensitive to outliers

<a name='ex-1'></a>
### Exercise 1 - K-Means Clustering

**Your Task:**

Implement the `apply_kmeans` function that:
1. Applies K-Means clustering to the data
2. Computes and returns inertia (within-cluster sum of squares)
3. Returns cluster labels and centroids

**Instructions:**
- Create a `KMeans` object with specified parameters
- Fit the model to the data
- Extract labels, centroids, and inertia
- Visualize the results

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For K-Means:**
* Create model: `kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)`
* Fit: `kmeans.fit(X)`
* Get labels: `labels = kmeans.labels_`
* Get centroids: `centers = kmeans.cluster_centers_`
* Get inertia: `inertia = kmeans.inertia_`

**For Analysis:**
* Plot using provided `plot_clusters` function
* Compute silhouette: `silhouette_score(X, labels)`

</details>

In [ ]:
# GRADED FUNCTION: apply_kmeans
def apply_kmeans(X, n_clusters=4, random_state=42):
    """
    Apply K-Means clustering to data.
    
    Args:
        X: Feature matrix
        n_clusters: Number of clusters
        random_state: Random seed
    
    Returns:
        dict with keys: 'labels', 'centers', 'inertia', 'silhouette'
    """
    ### START CODE HERE ### (≈ 8 lines)
    # Create KMeans object
    kmeans = None
    
    # Fit to data
    
    # Get cluster labels
    labels = None
    
    # Get cluster centers
    centers = None
    
    # Get inertia
    inertia = None
    
    # Compute silhouette score
    sil_score = None
    ### END CODE HERE ###
    
    return {
        'labels': labels,
        'centers': centers,
        'inertia': inertia,
        'silhouette': sil_score
    }

In [ ]:
# Test K-Means on blobs dataset
result = apply_kmeans(X_blobs_scaled, n_clusters=4)

print(f"K-Means Results (K=4):")
print(f"Inertia: {result['inertia']:.4f}")
print(f"Silhouette Score: {result['silhouette']:.4f}")
print(f"\nCluster sizes: {np.bincount(result['labels'])}")

# Visualize
plot_clusters(X_blobs_scaled, result['labels'], result['centers'],
              title="K-Means Clustering (K=4)")

##### **Expected Output**
```
K-Means Results (K=4):
Inertia: XXX.XXXX
Silhouette Score: 0.6XXX

Cluster sizes: [XX XX XX XX]
```
You should see 4 well-separated clusters with a silhouette score above 0.6.

In [ ]:
unittests.exercise_1(apply_kmeans)

<a name='3'></a>
## 3 - Optimal K Selection

Choosing the right number of clusters (K) is crucial but challenging in unsupervised learning. Two popular methods:

### 1. Elbow Method

Plot inertia vs. K and look for an "elbow" where inertia decreases slowly:

- **Low K**: High inertia (poor fit)
- **Optimal K**: At the elbow point
- **High K**: Low inertia but overfitting

### 2. Silhouette Analysis

Measures how similar points are to their own cluster vs. other clusters:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

where:
- $a(i)$: Mean distance to points in same cluster
- $b(i)$: Mean distance to points in nearest different cluster

**Interpretation**:
- **s ≈ 1**: Point is well-clustered
- **s ≈ 0**: Point is on cluster boundary
- **s < 0**: Point may be in wrong cluster

**Best K**: Maximizes average silhouette score with balanced cluster sizes.

<a name='ex-2'></a>
### Exercise 2 - Elbow Method and Silhouette Analysis

**Your Task:**

Implement the `find_optimal_k` function that:
1. Tests different values of K (from 2 to max_k)
2. Computes inertia and silhouette score for each K
3. Plots elbow curve and silhouette scores
4. Returns the optimal K based on silhouette score

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For loop over K values:**
* Loop: `for k in range(2, max_k + 1):`
* Use your `apply_kmeans` function
* Store inertia: `inertias.append(result['inertia'])`
* Store silhouette: `silhouettes.append(result['silhouette'])`

**For plotting:**
* Two subplots: `fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))`
* Elbow plot: `ax1.plot(k_range, inertias, marker='o')`
* Silhouette plot: `ax2.plot(k_range, silhouettes, marker='o')`

**For optimal K:**
* Find max silhouette: `optimal_k = k_range[np.argmax(silhouettes)]`

</details>

In [ ]:
# GRADED FUNCTION: find_optimal_k
def find_optimal_k(X, max_k=10):
    """
    Find optimal K using elbow method and silhouette analysis.
    
    Args:
        X: Feature matrix
        max_k: Maximum K to test
    
    Returns:
        optimal_k: Best K value based on silhouette score
    """
    inertias = []
    silhouettes = []
    k_range = range(2, max_k + 1)
    
    ### START CODE HERE ### (≈ 5 lines)
    # Loop over K values
    for k in k_range:
        # Apply K-Means
        result = None
        
        # Store inertia
        
        # Store silhouette score
        
    ### END CODE HERE ###
    
    # Plot results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Elbow plot
    ax1.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Number of Clusters (K)', fontsize=12)
    ax1.set_ylabel('Inertia', fontsize=12)
    ax1.set_title('Elbow Method', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Silhouette plot
    ax2.plot(k_range, silhouettes, 'ro-', linewidth=2, markersize=8)
    ax2.set_xlabel('Number of Clusters (K)', fontsize=12)
    ax2.set_ylabel('Silhouette Score', fontsize=12)
    ax2.set_title('Silhouette Analysis', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Mark optimal K
    optimal_k = k_range[np.argmax(silhouettes)]
    ax2.axvline(x=optimal_k, color='green', linestyle='--', linewidth=2,
                label=f'Optimal K={optimal_k}')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    return optimal_k

In [ ]:
# Find optimal K for blobs dataset
optimal_k = find_optimal_k(X_blobs_scaled, max_k=8)
print(f"\nOptimal K (based on silhouette): {optimal_k}")

# Apply with optimal K
result_optimal = apply_kmeans(X_blobs_scaled, n_clusters=optimal_k)
print(f"\nOptimal Clustering Results:")
print(f"Silhouette Score: {result_optimal['silhouette']:.4f}")

# Visualize silhouette plot
plot_silhouette(X_blobs_scaled, result_optimal['labels'], optimal_k)

##### **Expected Output**
```
Optimal K (based on silhouette): 4

Optimal Clustering Results:
Silhouette Score: 0.6XXX
```
The elbow plot should show a clear bend at K=4, and silhouette score should be highest at K=4.

In [ ]:
unittests.exercise_2(find_optimal_k)

<a name='4'></a>
## 4 - Gaussian Mixture Models (GMM)

**Gaussian Mixture Models** are probabilistic clustering algorithms that assume data comes from a mixture of Gaussian distributions.

### Key Differences from K-Means:

| Aspect | K-Means | GMM |
|--------|---------|-----|
| **Assignment** | Hard (0 or 1) | Soft (probabilities) |
| **Cluster Shape** | Spherical | Elliptical |
| **Model** | Distance-based | Probabilistic |
| **Output** | Cluster labels | Cluster probabilities |

### Mathematical Formulation:

**Probability of point x belonging to cluster k**:
$$p(x|k) = \mathcal{N}(x|\mu_k, \Sigma_k) = \frac{1}{(2\pi)^{d/2}|\Sigma_k|^{1/2}} \exp\left(-\frac{1}{2}(x-\mu_k)^T\Sigma_k^{-1}(x-\mu_k)\right)$$

**Mixture model**:
$$p(x) = \sum_{k=1}^{K} \pi_k \cdot \mathcal{N}(x|\mu_k, \Sigma_k)$$

where:
- $\mu_k$: Mean of cluster k
- $\Sigma_k$: Covariance matrix of cluster k
- $\pi_k$: Mixing coefficient (prior probability)

### Advantages:
Soft assignments (probabilistic)
Handles elliptical clusters
Can estimate cluster covariances
Provides probability of cluster membership

<a name='ex-3'></a>
### Exercise 3 - Implement GMM Clustering

**Your Task:**

Implement the `apply_gmm` function that:
1. Creates and fits a Gaussian Mixture Model
2. Predicts cluster assignments (hard labels)
3. Computes cluster probabilities (soft assignments)
4. Returns model information including BIC and AIC

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For GMM:**
* Create model: `gmm = GaussianMixture(n_components=k, random_state=42)`
* Fit: `gmm.fit(X)`
* Predict labels: `labels = gmm.predict(X)`
* Get probabilities: `probs = gmm.predict_proba(X)`
* BIC: `bic = gmm.bic(X)`
* AIC: `aic = gmm.aic(X)`

**For analysis:**
* Silhouette: `silhouette_score(X, labels)`
* Means: `gmm.means_`
* Covariances: `gmm.covariances_`

</details>

In [ ]:
# GRADED FUNCTION: apply_gmm
def apply_gmm(X, n_components=4, random_state=42):
    """
    Apply Gaussian Mixture Model clustering.
    
    Args:
        X: Feature matrix
        n_components: Number of mixture components
        random_state: Random seed
    
    Returns:
        dict with keys: 'labels', 'probabilities', 'gmm', 'bic', 'aic', 'silhouette'
    """
    ### START CODE HERE ### (≈ 10 lines)
    # Create GMM object
    gmm = None
    
    # Fit to data
    
    # Get cluster labels (hard assignment)
    labels = None
    
    # Get cluster probabilities (soft assignment)
    probabilities = None
    
    # Compute BIC and AIC
    bic = None
    aic = None
    
    # Compute silhouette score
    sil_score = None
    ### END CODE HERE ###
    
    return {
        'labels': labels,
        'probabilities': probabilities,
        'gmm': gmm,
        'bic': bic,
        'aic': aic,
        'silhouette': sil_score
    }

In [ ]:
# Test GMM on blobs dataset
gmm_result = apply_gmm(X_blobs_scaled, n_components=4)

print(f"GMM Results (K=4):")
print(f"BIC: {gmm_result['bic']:.4f}")
print(f"AIC: {gmm_result['aic']:.4f}")
print(f"Silhouette Score: {gmm_result['silhouette']:.4f}")
print(f"\nCluster sizes: {np.bincount(gmm_result['labels'])}")

# Show probability distribution for first 5 points
print(f"\nCluster probabilities for first 5 points:")
print(gmm_result['probabilities'][:5])

# Visualize with confidence ellipses
plot_gmm_ellipses(gmm_result['gmm'], X_blobs_scaled, gmm_result['labels'])

##### **Expected Output**
```
GMM Results (K=4):
BIC: XXX.XXXX
AIC: XXX.XXXX
Silhouette Score: 0.6XXX

Cluster sizes: [XX XX XX XX]

Cluster probabilities for first 5 points:
[[0.XXX 0.XXX 0.XXX 0.XXX]
 ...
```
Each row should sum to 1.0. You should see confidence ellipses around cluster centers.

In [ ]:
unittests.exercise_3(apply_gmm)

<a name='5'></a>
## 5 - Hierarchical Clustering

**Hierarchical clustering** builds a tree of clusters (dendrogram) without requiring K to be specified upfront.

### Two Approaches:

1. **Agglomerative (Bottom-Up)**:
   - Start: Each point is its own cluster
   - Repeatedly: Merge closest clusters
   - End: All points in one cluster

2. **Divisive (Top-Down)**:
   - Start: All points in one cluster
   - Repeatedly: Split furthest clusters
   - End: Each point in its own cluster

### Linkage Methods (how to measure cluster distance):

| Method | Description | Use When |
|--------|-------------|----------|
| **Single** | Min distance between points | Elongated clusters |
| **Complete** | Max distance between points | Compact clusters |
| **Average** | Average distance | Balanced |
| **Ward** | Minimizes variance | Most common, balanced |

**Ward Linkage** (most popular):
$$d(C_i, C_j) = \frac{|C_i| \cdot |C_j|}{|C_i| + |C_j|} ||\mu_i - \mu_j||^2$$

### Advantages:
No need to specify K upfront
Produces interpretable dendrogram
Can cut tree at any level
Deterministic results

### Disadvantages:
❌ Computationally expensive: O(n³)
❌ Cannot undo merges
❌ Sensitive to outliers

<a name='ex-4'></a>
### Exercise 4 - Build Dendrogram and Compare Strategies

**Your Task:**

Implement two functions:

1. `apply_hierarchical`: Apply agglomerative clustering with specified linkage
2. `compare_linkage_methods`: Compare different linkage strategies

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For hierarchical clustering:**
* Create model: `agg = AgglomerativeClustering(n_clusters=k, linkage='ward')`
* Fit and predict: `labels = agg.fit_predict(X)`
* Silhouette: `silhouette_score(X, labels)`

**For dendrogram:**
* Compute linkage: `Z = linkage(X, method='ward')`
* Plot: `dendrogram(Z, truncate_mode='lastp', p=30)`

**For comparison:**
* Loop over linkages: `for linkage_method in linkages:`
* Store results in dictionary

</details>

In [ ]:
# GRADED FUNCTION: apply_hierarchical
def apply_hierarchical(X, n_clusters=4, linkage_method='ward'):
    """
    Apply agglomerative hierarchical clustering.
    
    Args:
        X: Feature matrix
        n_clusters: Number of clusters
        linkage_method: 'ward', 'complete', 'average', or 'single'
    
    Returns:
        dict with keys: 'labels', 'silhouette'
    """
    ### START CODE HERE ### (≈ 5 lines)
    # Create AgglomerativeClustering object
    agg_clustering = None
    
    # Fit and predict
    labels = None
    
    # Compute silhouette score
    sil_score = None
    ### END CODE HERE ###
    
    return {
        'labels': labels,
        'silhouette': sil_score
    }

In [ ]:
# Apply hierarchical clustering with Ward linkage
hier_result = apply_hierarchical(X_blobs_scaled, n_clusters=4, linkage_method='ward')

print(f"Hierarchical Clustering (Ward linkage):")
print(f"Silhouette Score: {hier_result['silhouette']:.4f}")
print(f"Cluster sizes: {np.bincount(hier_result['labels'])}")

# Visualize
plot_clusters(X_blobs_scaled, hier_result['labels'],
              title="Hierarchical Clustering (Ward)")

# Plot dendrogram
plt.figure(figsize=(12, 6))
Z = linkage(X_blobs_scaled, method='ward')
dendrogram(Z, truncate_mode='lastp', p=30)
plt.title('Hierarchical Clustering Dendrogram (Ward)', fontsize=14, fontweight='bold')
plt.xlabel('Sample Index (or Cluster Size)', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_4(apply_hierarchical)

#### `compare_linkage_methods` - Compare All Linkage Strategies

Implement `compare_linkage_methods` that applies `apply_hierarchical` with each of the four
linkage methods (`'ward'`, `'complete'`, `'average'`, `'single'`) and returns a dictionary
mapping each method name to its result.

In [ ]:
# GRADED FUNCTION: compare_linkage_methods
def compare_linkage_methods(X, n_clusters=4):
    """
    Compare different linkage methods for hierarchical clustering.
    
    Args:
        X: Feature matrix
        n_clusters: Number of clusters
    
    Returns:
        results: dict mapping linkage method to results
    """
    linkages = ['ward', 'complete', 'average', 'single']
    results = {}
    
    ### START CODE HERE ### (≈ 3 lines)
    for linkage_method in linkages:
        # Apply hierarchical clustering
        result = None
        # Store result
        
    ### END CODE HERE ###
    
    return results

In [ ]:
# Compare linkage methods
comparison = compare_linkage_methods(X_blobs_scaled, n_clusters=4)

print("Comparison of Linkage Methods:\n")
print(f"{'Method':<12} {'Silhouette':<12}")
print("-" * 24)
for method, result in comparison.items():
    print(f"{method:<12} {result['silhouette']:<12.4f}")

# Visualize all methods
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, (method, result) in enumerate(comparison.items()):
    ax = axes[idx]
    scatter = ax.scatter(X_blobs_scaled[:, 0], X_blobs_scaled[:, 1], 
                        c=result['labels'], cmap='viridis', 
                        alpha=0.6, edgecolors='k', s=50)
    ax.set_title(f"{method.capitalize()} Linkage\nSilhouette: {result['silhouette']:.3f}",
                fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    plt.colorbar(scatter, ax=ax, label='Cluster')

plt.tight_layout()
plt.show()

##### **Expected Output**
```
Hierarchical Clustering (Ward linkage):
Silhouette Score: 0.6XXX
Cluster sizes: [XX XX XX XX]

Comparison of Linkage Methods:

Method       Silhouette  
------------------------
ward         0.6XXX      
complete     0.6XXX      
average      0.6XXX      
single       0.XXX       
```
Ward linkage should perform best. Single linkage typically performs worst due to chaining effect.

In [ ]:
unittests.exercise_5(compare_linkage_methods)

<a name='6'></a>
## 6 - Clustering Evaluation

Evaluating clustering quality is challenging without ground truth labels. We use two types of metrics:

### 1. Internal Metrics (No True Labels Needed)

**Silhouette Score** (already covered):
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$
Range: [-1, 1], Higher is better

**Davies-Bouldin Index**:
$$DB = \frac{1}{K} \sum_{i=1}^{K} \max_{j \neq i} \frac{\sigma_i + \sigma_j}{d(c_i, c_j)}$$
Lower is better (measures cluster separation)

**Calinski-Harabasz Index** (Variance Ratio):
$$CH = \frac{\text{Between-cluster variance}}{\text{Within-cluster variance}} \times \frac{n - K}{K - 1}$$
Higher is better

### 2. External Metrics (Require True Labels)

**Adjusted Rand Index (ARI)**:
$$ARI = \frac{\text{RI} - \text{Expected RI}}{\max(\text{RI}) - \text{Expected RI}}$$
Range: [-1, 1], 1 = perfect match, 0 = random

**Normalized Mutual Information (NMI)**:
$$NMI(Y, C) = \frac{2 \times I(Y; C)}{H(Y) + H(C)}$$
Range: [0, 1], 1 = perfect match

where $I$ is mutual information, $H$ is entropy.

<a name='ex-5'></a>
### Exercise 5 - Apply Clustering Metrics

**Your Task:**

Implement the `evaluate_clustering` function that:
1. Computes internal metrics (silhouette, Davies-Bouldin, Calinski-Harabasz)
2. If true labels available, computes external metrics (ARI, NMI)
3. Returns comprehensive evaluation dictionary

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For internal metrics:**
* Silhouette: `silhouette_score(X, labels)`
* Davies-Bouldin: `davies_bouldin_score(X, labels)` (from sklearn.metrics)
* Calinski-Harabasz: `calinski_harabasz_score(X, labels)` (from sklearn.metrics)

**For external metrics:**
* ARI: `adjusted_rand_score(y_true, labels)`
* NMI: `normalized_mutual_info_score(y_true, labels)`

**Check if true labels exist:**
* `if y_true is not None:`

</details>

In [ ]:
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score

# GRADED FUNCTION: evaluate_clustering
def evaluate_clustering(X, labels, y_true=None):
    """
    Evaluate clustering using internal and external metrics.
    
    Args:
        X: Feature matrix
        labels: Predicted cluster labels
        y_true: True labels (optional, for external metrics)
    
    Returns:
        dict with evaluation metrics
    """
    metrics = {}
    
    ### START CODE HERE ### (≈ 10 lines)
    # Internal metrics
    metrics['silhouette'] = None
    metrics['davies_bouldin'] = None  # Lower is better
    metrics['calinski_harabasz'] = None  # Higher is better
    
    # External metrics (if true labels available)
    if y_true is not None:
        metrics['ari'] = None
        metrics['nmi'] = None
    ### END CODE HERE ###
    
    return metrics

In [ ]:
# Evaluate different algorithms on Iris dataset
print("=" * 60)
print("CLUSTERING EVALUATION ON IRIS DATASET")
print("=" * 60)

# Apply different clustering methods
kmeans_iris = apply_kmeans(X_iris_scaled, n_clusters=3)
gmm_iris = apply_gmm(X_iris_scaled, n_components=3)
hier_iris = apply_hierarchical(X_iris_scaled, n_clusters=3)

# Evaluate each method
methods = {
    'K-Means': kmeans_iris['labels'],
    'GMM': gmm_iris['labels'],
    'Hierarchical': hier_iris['labels']
}

results_df = []
for name, labels in methods.items():
    metrics = evaluate_clustering(X_iris_scaled, labels, y_iris)
    metrics['Method'] = name
    results_df.append(metrics)

# Create comparison table
df = pd.DataFrame(results_df)
df = df[['Method', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'ari', 'nmi']]

print("\n")
print(df.to_string(index=False))
print("\n")

# Find best method for each metric
print("Best Performers:")
print(f"  Highest Silhouette: {df.loc[df['silhouette'].idxmax(), 'Method']} ({df['silhouette'].max():.4f})")
print(f"  Lowest Davies-Bouldin: {df.loc[df['davies_bouldin'].idxmin(), 'Method']} ({df['davies_bouldin'].min():.4f})")
print(f"  Highest Calinski-Harabasz: {df.loc[df['calinski_harabasz'].idxmax(), 'Method']} ({df['calinski_harabasz'].max():.2f})")
print(f"  Highest ARI: {df.loc[df['ari'].idxmax(), 'Method']} ({df['ari'].max():.4f})")
print(f"  Highest NMI: {df.loc[df['nmi'].idxmax(), 'Method']} ({df['nmi'].max():.4f})")

##### **Expected Output**
```
============================================================
CLUSTERING EVALUATION ON IRIS DATASET
============================================================

      Method  silhouette  davies_bouldin  calinski_harabasz      ari      nmi
     K-Means      0.5XXX          0.6XXX             XXX.XX   0.7XXX   0.7XXX
         GMM      0.5XXX          0.6XXX             XXX.XX   0.7XXX   0.7XXX
Hierarchical      0.5XXX          0.6XXX             XXX.XX   0.7XXX   0.7XXX

Best Performers:
  ...
```
All methods should perform reasonably well on Iris. ARI and NMI should be > 0.7.

In [ ]:
unittests.exercise_6(evaluate_clustering)

<a name='7'></a>
## 7 - From-Scratch Implementations

Understanding the internals of clustering algorithms helps you:
- Customize algorithms for specific needs
- Debug issues when algorithms fail
- Optimize for performance
- Understand convergence and initialization

We'll implement:
1. **K-Means from scratch** with Lloyd's algorithm
2. **GMM from scratch** using Expectation-Maximization

<a name='ex-6'></a>
### Exercise 6 - K-Means from Scratch

**Your Task:**

Implement K-Means algorithm from scratch:

1. `initialize_centroids`: Randomly select K data points as initial centroids
2. `assign_clusters`: Assign each point to nearest centroid
3. `update_centroids`: Recompute centroids as mean of assigned points
4. `kmeans_from_scratch`: Main loop combining above steps

**Algorithm Steps:**
```
1. Initialize K centroids randomly
2. Repeat until convergence:
   a. Assign each point to nearest centroid
   b. Update centroids as mean of assigned points
   c. Check if centroids changed
3. Return final labels and centroids
```

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For initialization:**
* Random indices: `indices = np.random.choice(len(X), k, replace=False)`
* Select points: `centroids = X[indices]`

**For assignment:**
* Compute distances: `distances = cdist(X, centroids, metric='euclidean')`
* Find nearest: `labels = np.argmin(distances, axis=1)`

**For update:**
* Loop over clusters: `for k in range(n_clusters):`
* Get cluster points: `cluster_points = X[labels == k]`
* Compute mean: `new_centroids[k] = cluster_points.mean(axis=0)`

**For convergence:**
* Check if equal: `if np.allclose(centroids, new_centroids):`
* Or use max iterations

</details>

In [ ]:
# GRADED FUNCTION: initialize_centroids
def initialize_centroids(X, k, random_state=42):
    """
    Initialize centroids by randomly selecting k data points.
    
    Args:
        X: Feature matrix (n_samples, n_features)
        k: Number of clusters
        random_state: Random seed
    
    Returns:
        centroids: Initial centroids (k, n_features)
    """
    np.random.seed(random_state)
    
    ### START CODE HERE ### (≈ 2 lines)
    # Randomly select k indices
    indices = None
    # Get corresponding points
    centroids = None
    ### END CODE HERE ###
    
    return centroids

In [ ]:
unittests.exercise_7(initialize_centroids)

#### `assign_clusters` - Assign Each Point to Nearest Centroid

Compute the Euclidean distance from each point to every centroid and assign each point to
the cluster of the nearest centroid.

In [ ]:
# GRADED FUNCTION: assign_clusters
def assign_clusters(X, centroids):
    """
    Assign each point to the nearest centroid.
    
    Args:
        X: Feature matrix (n_samples, n_features)
        centroids: Current centroids (k, n_features)
    
    Returns:
        labels: Cluster assignment for each point (n_samples,)
    """
    ### START CODE HERE ### (≈ 3 lines)
    # Compute distances from each point to each centroid
    distances = None
    # Assign to nearest centroid
    labels = None
    ### END CODE HERE ###
    
    return labels

In [ ]:
unittests.exercise_8(assign_clusters)

#### `update_centroids` - Recompute Centroids

Recompute each centroid as the mean of all points currently assigned to that cluster.
Handle empty clusters gracefully.

In [ ]:
# GRADED FUNCTION: update_centroids
def update_centroids(X, labels, k):
    """
    Update centroids as the mean of assigned points.
    
    Args:
        X: Feature matrix (n_samples, n_features)
        labels: Current cluster assignments (n_samples,)
        k: Number of clusters
    
    Returns:
        centroids: Updated centroids (k, n_features)
    """
    n_features = X.shape[1]
    centroids = np.zeros((k, n_features))
    
    ### START CODE HERE ### (≈ 4 lines)
    for cluster in range(k):
        # Get points in this cluster
        cluster_points = None
        # Compute mean (handle empty clusters)
        if len(cluster_points) > 0:
            centroids[cluster] = None
    ### END CODE HERE ###
    
    return centroids

In [ ]:
unittests.exercise_9(update_centroids)

#### `kmeans_from_scratch` - Full K-Means Loop

Combine the three steps above into the complete K-Means algorithm:
initialize centroids, then repeatedly assign and update until convergence or `max_iters` is reached.

In [ ]:
# GRADED FUNCTION: kmeans_from_scratch
def kmeans_from_scratch(X, k, max_iters=100, random_state=42):
    """
    K-Means clustering from scratch.
    
    Args:
        X: Feature matrix
        k: Number of clusters
        max_iters: Maximum iterations
        random_state: Random seed
    
    Returns:
        dict with 'labels', 'centroids', 'inertia', 'n_iters'
    """
    ### START CODE HERE ### (≈ 15 lines)
    # Initialize centroids
    centroids = None
    
    # Main loop
    for iteration in range(max_iters):
        # Assignment step
        labels = None
        
        # Update step
        new_centroids = None
        
        # Check convergence
        if np.allclose(centroids, new_centroids):
            centroids = new_centroids
            break
        
        centroids = new_centroids
    
    # Compute final inertia
    distances = None
    min_distances = None
    inertia = None
    ### END CODE HERE ###
    
    return {
        'labels': labels,
        'centroids': centroids,
        'inertia': inertia,
        'n_iters': iteration + 1
    }

In [ ]:
unittests.exercise_10(kmeans_from_scratch)

In [ ]:
# Test K-Means from scratch
scratch_result = kmeans_from_scratch(X_blobs_scaled, k=4, random_state=42)

print("K-Means from Scratch:")
print(f"Converged in {scratch_result['n_iters']} iterations")
print(f"Inertia: {scratch_result['inertia']:.4f}")
print(f"Cluster sizes: {np.bincount(scratch_result['labels'])}")

# Compare with sklearn
sklearn_result = apply_kmeans(X_blobs_scaled, n_clusters=4)
print(f"\nSklearn K-Means Inertia: {sklearn_result['inertia']:.4f}")
print(f"Difference: {abs(scratch_result['inertia'] - sklearn_result['inertia']):.6f}")

# Visualize
plot_clusters(X_blobs_scaled, scratch_result['labels'], 
              scratch_result['centroids'],
              title="K-Means from Scratch")

##### **Expected Output**
```
K-Means from Scratch:
Converged in X iterations
Inertia: XXX.XXXX
Cluster sizes: [XX XX XX XX]

Sklearn K-Means Inertia: XXX.XXXX
Difference: 0.XXXXXX
```
Results should be very close to sklearn implementation (small differences due to initialization).

<a name='ex-7'></a>
### Exercise 7 - GMM from Scratch (Expectation-Maximization)

**Your Task:**

Implement Gaussian Mixture Model using the Expectation-Maximization (EM) algorithm:

**E-Step (Expectation)**:
Compute probability that each point belongs to each cluster:
$$\gamma_{ik} = \frac{\pi_k \cdot \mathcal{N}(x_i|\mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j \cdot \mathcal{N}(x_i|\mu_j, \Sigma_j)}$$

**M-Step (Maximization)**:
Update parameters based on responsibilities:
$$\pi_k = \frac{1}{n}\sum_{i=1}^{n} \gamma_{ik}$$
$$\mu_k = \frac{\sum_{i=1}^{n} \gamma_{ik} x_i}{\sum_{i=1}^{n} \gamma_{ik}}$$
$$\Sigma_k = \frac{\sum_{i=1}^{n} \gamma_{ik} (x_i - \mu_k)(x_i - \mu_k)^T}{\sum_{i=1}^{n} \gamma_{ik}}$$

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For initialization:**
* Use K-Means to initialize: `kmeans_result = kmeans_from_scratch(X, k)`
* Means: `means = kmeans_result['centroids']`
* Covariances: `np.cov(X.T) * np.eye(n_features)` for each cluster
* Weights: `np.ones(k) / k`

**For E-step:**
* PDF: `multivariate_normal.pdf(X, mean=means[k], cov=covs[k])`
* Weighted: `weights[k] * pdf`
* Normalize: `responsibilities[k] / responsibilities.sum(axis=0)`

**For M-step:**
* Total responsibility: `Nk = responsibilities[k].sum()`
* Update weight: `weights[k] = Nk / n_samples`
* Update mean: `means[k] = (X * responsibilities[k]).sum(axis=0) / Nk`
* Update covariance: Similar weighted computation

**Note**: Add small value (1e-6) to diagonal of covariance for numerical stability

</details>

In [ ]:
# GRADED FUNCTION: gmm_from_scratch
def gmm_from_scratch(X, k, max_iters=100, tol=1e-4, random_state=42):
    """
    Gaussian Mixture Model using Expectation-Maximization.
    
    Args:
        X: Feature matrix (n_samples, n_features)
        k: Number of components
        max_iters: Maximum iterations
        tol: Convergence tolerance
        random_state: Random seed
    
    Returns:
        dict with 'labels', 'means', 'covariances', 'weights', 'responsibilities'
    """
    n_samples, n_features = X.shape
    
    ### START CODE HERE ### (≈ 40 lines)
    # Initialize using K-Means
    kmeans_result = None
    means = None
    
    # Initialize covariances as spherical
    covariances = np.array([np.eye(n_features) for _ in range(k)])
    
    # Initialize weights (mixing coefficients)
    weights = None
    
    prev_log_likelihood = -np.inf
    
    for iteration in range(max_iters):
        # E-Step: Compute responsibilities
        responsibilities = np.zeros((k, n_samples))
        
        for j in range(k):
            # Compute probability density for cluster j
            responsibilities[j] = None
        
        # Normalize responsibilities
        responsibilities /= responsibilities.sum(axis=0)
        
        # M-Step: Update parameters
        for j in range(k):
            # Total responsibility for cluster j
            Nj = None
            
            # Update weights
            weights[j] = None
            
            # Update means
            means[j] = None
            
            # Update covariances
            diff = X - means[j]
            covariances[j] = None
            # Add small value to diagonal for numerical stability
            covariances[j] += np.eye(n_features) * 1e-6
        
        # Compute log likelihood
        log_likelihood = 0
        for j in range(k):
            log_likelihood += (weights[j] * 
                             multivariate_normal.pdf(X, mean=means[j], 
                                                    cov=covariances[j]))
        log_likelihood = np.log(log_likelihood).sum()
        
        # Check convergence
        if abs(log_likelihood - prev_log_likelihood) < tol:
            break
        prev_log_likelihood = log_likelihood
    
    # Get final cluster assignments
    labels = None
    ### END CODE HERE ###
    
    return {
        'labels': labels,
        'means': means,
        'covariances': covariances,
        'weights': weights,
        'responsibilities': responsibilities,
        'n_iters': iteration + 1
    }

In [ ]:
unittests.exercise_11(gmm_from_scratch)

In [ ]:
# Test GMM from scratch
gmm_scratch = gmm_from_scratch(X_blobs_scaled, k=4, random_state=42)

print("GMM from Scratch:")
print(f"Converged in {gmm_scratch['n_iters']} iterations")
print(f"Mixing weights: {gmm_scratch['weights']}")
print(f"Cluster sizes: {np.bincount(gmm_scratch['labels'])}")

# Compare with sklearn
gmm_sklearn = apply_gmm(X_blobs_scaled, n_components=4)
print(f"\nComparison with sklearn:")
print(f"Scratch silhouette: {silhouette_score(X_blobs_scaled, gmm_scratch['labels']):.4f}")
print(f"Sklearn silhouette: {gmm_sklearn['silhouette']:.4f}")

# Visualize
plot_clusters(X_blobs_scaled, gmm_scratch['labels'],
              gmm_scratch['means'],
              title="GMM from Scratch")

# Show soft assignments for first 5 points
print(f"\nSoft cluster assignments (first 5 points):")
print(gmm_scratch['responsibilities'][:, :5].T)

##### **Expected Output**
```
GMM from Scratch:
Converged in XX iterations
Mixing weights: [0.2XX 0.2XX 0.2XX 0.2XX]
Cluster sizes: [XX XX XX XX]

Comparison with sklearn:
Scratch silhouette: 0.6XXX
Sklearn silhouette: 0.6XXX

Soft cluster assignments (first 5 points):
[[0.XXX 0.XXX 0.XXX 0.XXX]
 ...
```
Each row should sum to 1.0. Results should be comparable to sklearn.

<a name='8'></a>
## 8 - Conclusion

**Congratulations!** You've completed the Clustering Algorithms assignment!

### What You've Learned:

1. **K-Means Clustering**: Implemented partitional clustering and analyzed inertia
2. **Optimal K Selection**: Used elbow method and silhouette analysis
3. **Gaussian Mixture Models**: Applied soft clustering with probability assignments
4. **Hierarchical Clustering**: Built dendrograms and compared linkage strategies
5. **Clustering Evaluation**: Used internal and external metrics
6. **K-Means from Scratch**: Implemented initialization, assignment, and update steps
7. **GMM from Scratch**: Implemented Expectation-Maximization algorithm

### Key Takeaways:

**Algorithm Selection:**
- **K-Means**: Fast, simple, works for spherical clusters
- **GMM**: Flexible cluster shapes, probabilistic assignments
- **Hierarchical**: No need to specify K, creates interpretable dendrograms
- **DBSCAN**: Handles arbitrary shapes, discovers outliers

**Best Practices:**
- Always standardize features before clustering
- Use multiple methods to determine optimal K
- Evaluate using multiple metrics
- Visualize results when possible (2D/3D)
- Be aware of algorithm assumptions and limitations

**Common Pitfalls:**
- Forgetting to scale features
- Using K-Means on non-spherical clusters
- Not validating cluster quality
- Ignoring initialization sensitivity
- Over-interpreting clusters in high dimensions

### Real-World Applications:

**Business:**
- Customer segmentation for targeted marketing
- Product recommendation systems
- Fraud detection

**Healthcare:**
- Patient stratification
- Disease subtype discovery
- Medical image segmentation

**Technology:**
- Image compression
- Document clustering
- Anomaly detection
- Social network analysis

### Next Steps:

1. **Explore Advanced Methods**: DBSCAN, HDBSCAN, Spectral Clustering
2. **Learn Dimensionality Reduction**: Combine with PCA, t-SNE, UMAP
3. **Apply to Real Data**: Work on Kaggle datasets
4. **Study Deep Clustering**: Autoencoders, Deep Embedded Clustering
5. **Master Interpretability**: SHAP values, cluster profiling

**Excellent work! You're now equipped to discover hidden patterns in unlabeled data!**